# LoCoMo cross-system retrieval

Headline benchmark: direct retrieval on upstream `locomo10.json` (10
conversations, 5,882 facts, 1,536 questions, top-k = 10). Each system is
seeded with the same conversation turns and scored against LoCoMo evidence
IDs. memd is compared against `mem0` and `SuperLocalMemory`.

Data: `evals/notebooks/data/locomo_2026-05-22.json` (summary slice of the
checked-in `evals/benchmarks/locomo/results/` report). Figures are written
to `docs/figures/` for the benchmark writeup.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import matplotlib.pyplot as plt
import numpy as np
from memd_plotting import (apply_house_style, color_for, label_bars,
                            load_json, locomo_systems, figures_dir)
apply_house_style()
FIG = figures_dir()
report = load_json('data/locomo_2026-05-22.json')
systems = locomo_systems(report)
order = ['memd', 'mem0', 'superlocalmemory']
systems.sort(key=lambda s: order.index(s['name']) if s['name'] in order else 99)
[(s['display'], round(s['mrr_at_10'], 3)) for s in systems]

## Figure 1 — retrieval quality (MRR@10 and Hit@k)

Grouped horizontal bars: the single quality summary readers care about.
memd leads on every metric.

In [ ]:
metrics = [('mrr_at_10', 'MRR@10'), ('hit1', 'Hit@1'), ('hit3', 'Hit@3'), ('hit10', 'Hit@10')]
names = [s['display'] for s in systems]
y = np.arange(len(metrics))
h = 0.78 / len(systems)
fig, ax = plt.subplots(figsize=(8.2, 4.6))
for i, s in enumerate(systems):
    vals = [s[k] for k, _ in metrics]
    offset = (i - (len(systems) - 1) / 2) * h
    bars = ax.barh(y + offset, vals, height=h, color=color_for(s['name']),
                   label=s['display'], edgecolor='white', linewidth=0.6)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_width() + 0.006, bar.get_y() + bar.get_height()/2,
                f'{v:.3f}', va='center', ha='left', fontsize=8, color='#333333')
ax.set_yticks(y)
ax.set_yticklabels([lbl for _, lbl in metrics])
ax.invert_yaxis()
ax.set_xlim(0, 0.78)
ax.set_xlabel('score (higher is better)')
ax.set_title('LoCoMo retrieval quality by system')
# Anchor the legend in the empty upper-right (MRR@10/Hit@1 rows top out
# near 0.42), clear of the long Hit@10 bars at the bottom.
ax.legend(loc='upper right', ncol=1, fontsize=9, borderaxespad=0.6)
ax.grid(axis='y', visible=False)
fig.tight_layout()
fig.savefig(FIG / 'locomo_quality.png')
fig.savefig(FIG / 'locomo_quality.svg')
plt.show()

## Figure 2 — quality vs. search latency

The other axis that matters operationally: memd is both the most accurate
and among the fastest to query. Bubble area is proportional to p95 search
latency; the x-axis is mean search latency (log scale — SuperLocalMemory is
an order of magnitude slower).

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 4.8))
for s in systems:
    ax.scatter(s['avg_search_ms'], s['mrr_at_10'],
               s=max(s['p95_search_ms'], 1) * 1.2 + 60,
               color=color_for(s['name']), alpha=0.85, edgecolor='white', linewidth=1.2,
               zorder=3)
    ax.annotate(f"{s['display']}\nMRR {s['mrr_at_10']:.3f}, p95 {s['p95_search_ms']:.0f}ms",
                (s['avg_search_ms'], s['mrr_at_10']),
                textcoords='offset points', xytext=(12, 6), fontsize=9)
ax.set_xscale('log')
ax.set_xlabel('mean search latency (ms, log scale — lower is better)')
ax.set_ylabel('MRR@10 (higher is better)')
ax.set_title('LoCoMo: retrieval quality vs. query latency')
ax.set_ylim(0.33, 0.45)
ax.margins(x=0.18)
fig.tight_layout()
fig.savefig(FIG / 'locomo_quality_latency.png')
fig.savefig(FIG / 'locomo_quality_latency.svg')
plt.show()

## Figure 3 — per-category MRR@10 heatmap

LoCoMo's four question categories stress different memory behaviours.
memd is strongest on category 2 (temporal/multi-hop) and competitive
everywhere; the heatmap shows where each system's advantage lies.

In [ ]:
cats = sorted({c for s in systems for c in s['per_category']}, key=int)
matrix = np.array([[s['per_category'].get(c, {}).get('mrr_at_10', np.nan) for c in cats]
                   for s in systems])
fig, ax = plt.subplots(figsize=(6.8, 3.6))
im = ax.imshow(matrix, cmap='BuPu', aspect='auto', vmin=0.2, vmax=0.55)
ax.set_xticks(range(len(cats)))
ax.set_xticklabels([f'Cat {c}' for c in cats])
ax.set_yticks(range(len(systems)))
ax.set_yticklabels([s['display'] for s in systems])
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        v = matrix[i, j]
        ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=9,
                color='white' if v > 0.42 else '#222222')
ax.set_title('LoCoMo MRR@10 by question category')
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('MRR@10')
ax.grid(False)
fig.tight_layout()
fig.savefig(FIG / 'locomo_per_category.png')
fig.savefig(FIG / 'locomo_per_category.svg')
plt.show()